# ADAPT-QAOA for Max-Cut with CUDA-Q

Standard QAOA alternates the cost unitary $e^{-i\gamma H_C}$ with a **fixed** mixer $e^{-i\beta \sum_i X_i}$. **ADAPT-QAOA** (Zhu *et al.*, *Adaptive quantum approximate optimization algorithm for solving combinatorial problems on a quantum computer*) instead **grows the circuit one layer at a time** and, at every layer, picks the mixer from a **pool of operators** that lowers the energy fastest.

Each ADAPT step:

1. Prepare the current state $|\psi_k\rangle$ and apply a small cost-unitary step $e^{-i\gamma_0 H_C}$.
2. For every pool operator $A_j$, compute the energy gradient $g_j = -i\,\langle [H_C, A_j]\rangle$.
3. If $\lVert g \rVert$ is below a threshold, stop. Otherwise, append a new layer whose mixer is the operator with the largest $|g_j|$.
4. Re-optimise **all** $\gamma$ and $\beta$ parameters with a classical optimizer (BFGS).
5. Stop when the energy change between steps is below `e_stop`.

ADAPT-QAOA usually reaches the solution with **far fewer layers** than standard QAOA.

**Kernel:** run `00_jupyter_kernel_from_container.ipynb` once to create the CUDA-Q kernels, then select **Python (CUDA-Q GPU)** in the top-right corner. To run on CPU instead, select **Python (CUDA-Q CPU)** and change the target in the first code cell to `cudaq.set_target("qpp-cpu")`.

## 1. Imports and target

In [ ]:
import numpy as np
import cudaq
from cudaq import spin
from scipy.optimize import minimize
import random

cudaq.set_target("nvidia", option="fp64")
print("Target:", cudaq.get_target().name)

## 2. The Max-Cut graph
We use a 5-node, 6-edge graph whose maximum cut is 5, i.e. a ground-state energy of $-5$. A larger 10-node example (true energy $-12$) is included as a comment: swap it in to try a harder problem.

In [ ]:
# Example 1 (larger)
# true_energy = -12.0
# nodes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# edges = [[0,4], [1,4], [0,6], [3,6], [5,6], [2,7], [1,8], [3,8], [5,8], [6,8], [7,8], [2,9], [8,9]]
# weight = [1.0] * len(edges)

# Example 2
true_energy = -5.0
nodes = [0, 1, 2, 3, 4]
edges = [[0, 1], [1, 2], [2, 3], [3, 0], [2, 4], [3, 4]]
weight = [1.0] * len(edges)

# One qubit per node
qubits_num = len(nodes)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)
nx.draw(G, with_labels=True, node_color="lightblue", node_size=700)
plt.show()

## 3. Stopping criteria and initial $\gamma$
- `threshold`: stop when the norm of the gradient vector falls below this value.
- `e_stop`: stop when the energy change between two ADAPT steps, $|E_k - E_{k-1}|$, falls below this value.
- `init_gamma`: the small cost-unitary angle $\gamma_0$ used when computing the gradients, and the initial $\gamma$ for every new layer.

In [ ]:
E_prev = 0.0
threshold = 1e-3   # stop if the norm of the gradient is smaller than this value
e_stop = 1e-7      # stop if dE = |current energy - previous energy| <= e_stop

init_gamma = 0.01

## 4. Cost Hamiltonian
For every edge $(u, v)$ with weight $w_{uv}$:

$$H_C = \sum_{(u,v)} \tfrac{1}{2}\, w_{uv}\,(Z_u Z_v - I)$$

Each edge contributes $-w_{uv}$ when its two nodes are on opposite sides of the cut, so **minimising $H_C$ maximises the cut**.

To apply $e^{-i\gamma H_C}$ inside a kernel with `exp_pauli`, we split the Hamiltonian into its **coefficients** and **Pauli words**. Each word covers all qubits, e.g. `ZZIII`.

In [ ]:
def spin_ham(edges, weight):
    ham = 0
    count = 0
    for edge in range(len(edges)):
        qubitu = edges[edge][0]
        qubitv = edges[edge][1]
        # Add a term to the Hamiltonian for the edge (u,v)
        ham += 0.5 * (weight[count] * cudaq.spin.z(qubitu) * cudaq.spin.z(qubitv) -
                      weight[count] * cudaq.spin.i(qubitu) * cudaq.spin.i(qubitv))
        count += 1
    return ham


# Collect coefficients from a spin operator so we can pass them to a kernel
def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    result = []
    for term in ham:
        result.append(term.evaluate_coefficient())
    return result


# Collect Pauli words from a spin operator so we can pass them to a kernel
def term_words(ham: cudaq.SpinOperator) -> list[str]:
    # The kernel uses these words to apply exp_pauli to the entire state,
    # so each Pauli word must cover the entire space.
    result = []
    for term in ham:
        result.append(term.get_pauli_word(qubits_num))
    return result


ham = spin_ham(edges, weight)
ham_coef = term_coefficients(ham)
ham_word = term_words(ham)

print(ham)
print("Pauli words :", ham_word)

## 5. Mixer pool
The candidate mixers are:

- **single-qubit** $X_i$ for every qubit,
- the **standard QAOA mixer** $\sum_i X_i$,
- **two-qubit** operators $X_iX_j$, $Y_iY_j$, $Y_iZ_j$, $Z_iY_j$ for every pair $i<j$.

The two-qubit entangling mixers let ADAPT-QAOA find shorter circuits than the standard mixer alone.

In [ ]:
def qaoa_mixer(n):
    term = spin.x(0)
    for i in range(1, n):
        term = term + spin.x(i)
    pool = [term]
    return pool


def qaoa_single_x(n):
    pool = []
    for i in range(n):
        pool.append(cudaq.SpinOperator(spin.x(i)))
    return pool


def qaoa_double_ops(n):
    pool = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            pool.append(cudaq.SpinOperator(spin.x(i)) * cudaq.SpinOperator(spin.x(j)))
            pool.append(cudaq.SpinOperator(spin.y(i)) * cudaq.SpinOperator(spin.y(j)))
            pool.append(cudaq.SpinOperator(spin.y(i)) * cudaq.SpinOperator(spin.z(j)))
            pool.append(cudaq.SpinOperator(spin.z(i)) * cudaq.SpinOperator(spin.y(j)))
    return pool


def mixer_pool(qubits_num):
    return (
          qaoa_single_x(qubits_num)
        + qaoa_mixer(qubits_num)
        + qaoa_double_ops(qubits_num)
    )


pools = mixer_pool(qubits_num)
print('Number of pool operators: ', len(pools))

## 6. Gradient operators
The gradient of the energy with respect to adding mixer $A_j$ is the expectation value of the commutator $[H_C, A_j] = H_C A_j - A_j H_C$, multiplied by $-i$ (applied in the ADAPT loop). We build all commutators once.

In [ ]:
def commutator(pools, ham):
    com_op = []
    for i in range(len(pools)):
        op = pools[i]
        com_op.append(ham * op - op * ham)
    return com_op


grad_op = commutator(pools, ham)

## 7. Quantum kernels
- `initial_state`: the uniform superposition $|+\rangle^{\otimes n}$, the reference state $|\psi_{\text{ref}}\rangle$.
- `grad`: starts from a given state and applies $e^{-i\gamma_0 H_C}$. The gradients are measured on this state.
- `kernel_qaoa`: the full ADAPT-QAOA circuit. Each layer $p$ applies $e^{-i\gamma_p H_C}$, then the mixer(s) selected for that layer with angle $\beta_p$.

In [ ]:
@cudaq.kernel
def initial_state(qubits_num: int):
    qubits = cudaq.qvector(qubits_num)
    h(qubits)


state = cudaq.get_state(initial_state, qubits_num)


# Circuit used to compute the energy gradient with respect to the pool
@cudaq.kernel
def grad(state: cudaq.State, ham_word: list[cudaq.pauli_word], ham_coef: list[complex], init_gamma: float):
    q = cudaq.qvector(state)
    for i in range(len(ham_coef)):
        exp_pauli(init_gamma * ham_coef[i].real, q, ham_word[i])


# The QAOA circuit using the selected pool operator(s) with max gradient
@cudaq.kernel
def kernel_qaoa(qubits_num: int, ham_word: list[cudaq.pauli_word], ham_coef: list[complex],
                mixer_pool: list[list[cudaq.pauli_word]], gamma: list[float], beta: list[float], num_layer: int):
    qubits = cudaq.qvector(qubits_num)
    h(qubits)

    for p in range(num_layer):
        for i in range(len(ham_coef)):
            exp_pauli(gamma[p] * ham_coef[i].real, qubits, ham_word[i])

        for word in mixer_pool[p]:
            exp_pauli(beta[p], qubits, word)

## 8. ADAPT-QAOA loop
At every step we:

1. measure all gradients $-i\langle[H_C, A_j]\rangle$ on the current state and stop if their norm is below `threshold`;
2. collect the operators with the largest $|g_j|$ and choose one (with a fixed random seed when several tie);
3. add a new layer with that mixer, $\gamma = \gamma_0$ and $\beta = 0$;
4. re-optimise all $\gamma, \beta$ with BFGS;
5. stop if the energy changed by less than `e_stop`. Otherwise, store the new state and continue.

In [ ]:
beta = []
gamma = []

mixer_pool = []
mixer_pool_str = []
layer = []

istep = 1
while True:

    print('Step: ', istep)

    # Compute the gradient and find the mixer pool with the largest values.
    # If the norm is below the predefined threshold, stop the calculation.
    gradient_vec = []
    for op in grad_op:
        op = op * -1j
        gradient_vec.append(cudaq.observe(grad, op, state, ham_word, ham_coef, init_gamma).expectation())

    # Norm of the gradient vector
    norm = np.linalg.norm(np.array(gradient_vec))
    print('Norm of the gradient: ', norm)

    if norm <= threshold:
        print('\n', 'Final Result: ', '\n')
        print('Final mixer_pool: ', mixer_pool_str)
        print('Number of layers: ', len(layer))
        print('Number of mixer pool in each layer: ', layer)
        print('Final Energy: ', result_vqe.fun)
        break

    else:
        temp_pool = []
        tot_pool = 0

        max_grad = np.max(np.abs(gradient_vec))

        for i in range(len(pools)):
            if np.abs(gradient_vec[i]) >= max_grad:
                tot_pool += 1
                temp_pool.append(pools[i])

        # Seed the random number generator so the choice among
        # equally large gradients is reproducible at every step.
        random.seed(42)

        layer.append(1)
        random_mixer = random.choice(temp_pool)

        pool_added = []
        pool_added_str = []
        for term in random_mixer:
            pool_added.append(cudaq.pauli_word(term.get_pauli_word(qubits_num)))
            pool_added_str.append(term.get_pauli_word(qubits_num))

        mixer_pool.append(pool_added)
        mixer_pool_str.append(pool_added_str)

        print('Mixer pool at step', istep)
        print(mixer_pool_str)

        num_layer = len(layer)
        print('Number of layers: ', num_layer)
        print(layer)
        beta_count = layer[num_layer - 1]
        init_beta = [0.0] * beta_count
        beta = beta + init_beta
        gamma = gamma + [init_gamma]
        theta = gamma + beta

        def cost(theta):
            theta = theta.tolist()
            gamma = theta[:num_layer]
            beta = theta[num_layer:]
            energy = cudaq.observe(kernel_qaoa, ham, qubits_num, ham_word, ham_coef,
                                   mixer_pool, gamma, beta, num_layer).expectation()
            return energy

        result_vqe = minimize(cost, theta, method='BFGS', jac='2-point', tol=1e-5)
        print('Result from the step ', istep)
        print('Optimized Energy: ', result_vqe.fun)

        dE = np.abs(result_vqe.fun - E_prev)
        E_prev = result_vqe.fun
        theta = result_vqe.x.tolist()
        erg = result_vqe.fun

        print('dE= :', dE)
        gamma = theta[:num_layer]
        beta = theta[num_layer:]

        if dE <= e_stop:
            print('\n', 'Final Result: ', '\n')
            print('Final mixer_pool: ', mixer_pool_str)
            print('Number of layers: ', len(layer))
            print('Number of mixer pool in each layer: ', layer)
            print('Final Energy= ', erg)
            break

        else:
            # State of the current step, used for the next gradient
            state = cudaq.get_state(kernel_qaoa, qubits_num, ham_word, ham_coef, mixer_pool, gamma, beta, num_layer)
            istep += 1
            print('\n')

## 9. Sample the final circuit
Sample the optimised ADAPT-QAOA circuit. The most probable bitstring gives the partition: nodes with `0` on one side of the cut and nodes with `1` on the other. Compare the final energy with `true_energy`.

In [ ]:
count = cudaq.sample(kernel_qaoa, qubits_num, ham_word, ham_coef, mixer_pool, gamma, beta, num_layer,
                     shots_count=5000)
print('True energy           : ', true_energy)
print('The most probable max cut: ', count.most_probable())
print('All bitstrings from circuit sampling: ', count)

## 10. Plot the cut

In [ ]:
partition = count.most_probable()
color_map = ["red" if partition[i] == "1" else "blue" for i in nodes]
cut_edges = [(u, v) for u, v in edges if partition[u] != partition[v]]
same_edges = [(u, v) for u, v in edges if partition[u] == partition[v]]
print(f"Cuts: {len(cut_edges)}")

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(6, 6))
nx.draw_networkx_nodes(G, pos, node_color=color_map, edgecolors="black", node_size=1200)
nx.draw_networkx_edges(G, pos, edgelist=same_edges, width=2, edge_color="gray", alpha=0.5)
nx.draw_networkx_edges(G, pos, edgelist=cut_edges, width=3, edge_color="green", style="dashed")
nx.draw_networkx_labels(G, pos, font_color="white", font_weight="bold")
plt.title("ADAPT-QAOA Max-Cut (red vs blue), cut edges green dashed")
plt.axis("off")
plt.show()